# Data Science 2 - Deep Learning FINAL ASSIGNMENT

**FINE TUNING SCRIPT**

**Author: Márton Nagy**

**Instructor: Eduardo Arino de la Rubia**

Installing needed packages. There are some dependency issues, but these come just from the Kaggle environment.

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

Importing the needed packages.

In [ ]:
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          TrainingArguments, DataCollatorForLanguageModeling,
                          BitsAndBytesConfig, pipeline, Trainer)
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
import torch
import warnings
warnings.filterwarnings('ignore')
import random

First, we have to decide what dataset to use for fine tuning. I looked through the [Awesome Summarizion Datasets](https://github.com/edahanoam/Awesome-Summarization-Datasets) to find something that may be relevant for my use case. I found two options that seemed relevant: them Amazon reviews dataset and the Gigaword dataset. The first contains the titles and texts of reviews on Amazon. This is relevant in its topic, but looking through a few entries, I noticed that this is not the task I want to train the model for, as the review titles were mostly just a few word summary, not full sentences. The Gigaword dataset contains leads of news articles with the corresponding titles. These titles are full sentences, thus replicate more my actual task. Note however, that the topic of the Gigaword set may still be problematic, as these are rather factual articles, as opposed to the opinionated reviews I have. Nonetheless, I could not find any sets that had opinionated texts (such as reviews) with full sentence summaries, so I deemed the task allignment more important than the topic allignment. Thus I have chosen to work with the Gigaword set.

Note that for efficiency, I only used a very small part of it (5,000 observations). Still this may be just enough to prove that fine tuning a model to a specific task can have some benefits.

In [ ]:
dataset = load_dataset('gigaword', split='train[:5000]', trust_remote_code=True)
eval = load_dataset('gigaword', split='test[:250]')

Now I load the base, pre-trained tokenizer and model (in quantized version for efficiency).

In [ ]:
model_name = 'unsloth/Llama-3.2-1B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bnb_config,
                                             device_map="auto")

For training, I created some shorter prompt versions which will be added to the inputs randomly. I did this so that the model may learn that the expected output is not different based on the modality of the prompt.

In [ ]:
prompt_versions = [
    'Summarize this review in one first-person sentence (max 10 words): ',
    'Provide a 1-sentence, first-person summary (max 10 words) for the following review: '
    'Write a first-person, single-sentence summary (under 11 words) focusing on the main points of this review: ',
    'Condense the key impressions from this review into one first-person sentence (10 words max): ',
    'Generate a first-person, one-sentence summary (maximum 10 words) for this review: ',
    'Task: Summarize review. Constraints: 1 sentence, first-person, <=10 words. Review follows: ',
    'One-sentence, first-person summary (max 10 words) needed for this review: ',
    'Extract the main point as a first-person, single sentence (10 words limit) from this review: ',
    'Review summary: first-person, 1 sentence, max 10 words. '
    'Your task: Generate a one-sentence, first-person summary (max 10 words) for the review below. ',
    'SUMMARIZE THIS REVIEW IN ONE FIRST-PERSON SENTENCE (MAX 10 WORDS): ',
    'Do this right: Write a first-person, one-sentence review summary, limit is 10 words: ',
    'Your life depends on summarizing this review in one first-person sentence (max 10 words): ',
    'You will do an amazing job! Summarize the review in one first-person sentence (max 10 words): ',
    'I believe in you! Write a first-person, one-sentence summary (≤10 words) for this review: ',
    'Please help me! Summarize the review in a first-person, single sentence (max 10 words): ',
    'I’m counting on you: one first-person sentence (around 10 words) summarizing this review: ',
    'This is easy. Summarize the review in a first-person sentence (10 words max): ',
    'Surely you can manage this: first-person, single-sentence summary in 10 words: '
]

I create a preprocessing function as well that generates appropriate inputs, labels and attention masks from the dataset. It also augments the inputs with the previously discussed inputs, as well as the appropriate chat template that the Llama model was initially trained on.

In [ ]:
ignore_index = -100
eos_token = tokenizer.eos_token

def preprocess(example):
    chosen_prompt = random.choice(prompt_versions)
    summary_with_eos = example['summary'] + eos_token

    full_text = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
        '<|eot_id|><|start_header_id|>user<|end_header_id|>\n' +
        chosen_prompt + example['document'] +
        '<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n' +
        summary_with_eos
    )
    tokenized_full = tokenizer(
        full_text,
        truncation=True,
        padding='max_length',
        max_length=256,
        add_special_tokens=False
    )

    labels = list(tokenized_full['input_ids'])

    input_part_text = (
         '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
         '<|eot_id|><|start_header_id|>user<|end_header_id|>\n' +
         chosen_prompt + example['document'] +
         '<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n'
    )
    tokenized_input_part = tokenizer(
        input_part_text,
        add_special_tokens=False
    )
    input_part_len = len(tokenized_input_part['input_ids'])

    for i in range(input_part_len):
        if i < len(labels):
            labels[i] = ignore_index

    actual_seq_len = len(tokenized_full['input_ids'])
    for i in range(actual_seq_len):
      if tokenized_full['input_ids'][i] == tokenizer.pad_token_id:
          if i < len(labels):
             labels[i] = ignore_index

    if all(l == ignore_index for l in labels):
       print(f"Warning: Example might have been truncated before assistant response started. Full text length: {len(full_text)}")

    return {
        'input_ids': tokenized_full['input_ids'],
        'attention_mask': tokenized_full['attention_mask'],
        'labels': labels,
    }

Now we can configure the PEFT model with LORA. Note that I heavily relied on Google, documentations and ChatGPT to provide reasonable configurations.

In [ ]:
model = (
    prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False})
    )
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    inference_mode=False,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

Using the previously defined preprocessing function, I tokenize the training and evaluation datasets.

In [ ]:
tokenized_data = dataset.map(preprocess, remove_columns=dataset.column_names)
tokenized_eval = eval.map(preprocess, remove_columns=eval.column_names)

Now we can set up the training arguments and the trainer itself. Note that again, I heavily relied on Google, documentations and ChatGPT to provide reasonable configurations.

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,
    eval_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=2e-5,
    logging_steps=150,
    eval_strategy='steps',
    eval_steps=150,
    output_dir="./lora_output",
    save_total_limit=1,
    fp16=True,
    report_to="none"
)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized_data,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    args=training_args,
)

We can finally run the training! This took about an hour with a P100 GPU on Kaggle.

In [ ]:
trainer.train()

To have one fine-tuned model we can directly use for inference, I merge and unload the trained model.

In [ ]:
model = model.merge_and_unload()

Saving and then loading the fine tuned version.

In [ ]:
finetuned_name = 'llama-3.2-1B-instruct-finetuned'
model.save_pretrained(finetuned_name)
tokenizer.save_pretrained(finetuned_name)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(finetuned_name)
model = AutoModelForCausalLM.from_pretrained(finetuned_name)

First, I test that my fine tuned model actually works.

In [ ]:
pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map='cuda',
)
message = [{'role': 'user',
            'content': '''
Assume you are someone who have watched Pulp Fiction and have wrote the below review about it to IMDb.
You also have to provide a very short summary for the review, written in first-person voice.
This summary should summarize the review in one sentence.
The summary should concentrate on the main points and impressions expressed in the review.
Your task now is to generate this one-sentence summary for the review.
Only write the summary, do not write anything else. Write maximum 10 words.

Here is the review:
This movie will change you, as some have commented. It will bring you to a disgusting low. Definitely skip it, or you'll be sorry you didn't.It's a sickening wretchingly vile voyueristic wannabe false glorification.Skip it. In fact be happy to skip it. Sometimes I think we have a big pile of juvenile retards on IMDb, that only hope to vote for any movie along the lines of it's inherent jerry springer qualities and the judgement comes as a message about how low and worthless it can get.This movie is not entertaining, it is not nice, it is not realistic, it is racist, degrading, and stupid.Don't waste your money, your time, or your life.'''
},
]
outputs = pipe(
    message,
    max_new_tokens=25
)
print(outputs[0]['generated_text'][-1]['content'])

Having seen that the model works, I now import the reviews.

In [ ]:
import pandas as pd

In [ ]:
reviews = pd.read_csv('/kaggle/input/pulp-fiction-review-summarization/pulp_fiction_50_reviews_gemini.csv')

Defining the prompt versions (the same as I used before).

In [ ]:
prompt_versions = {
    'base' : '''
Assume you are someone who have watched Pulp Fiction and have wrote the below review about it to IMDb.
You also have to provide a very short summary for the review, written in first-person voice.
This summary should summarize the review in one sentence.
The summary should concentrate on the main points and impressions expressed in the review.
Your task now is to generate this one-sentence summary for the review.
Only write the summary, do not write anything else. Write maximum 10 words.

Here is the review:
''',
    'caps' : '''
ASSUME YOU are someone who HAVE WATCHED PULP FICTION and have WROTE THE BELOW REVIEW about it to IMDb.
You also HAVE TO PROVIDE A VERY SHORT SUMMARY for the review, written in FIRST-PERSON VOICE.
This SUMMARY should SUMMARIZE THE REVIEW IN ONE SENTENCE.
The SUMMARY should CONCENTRATE on the MAIN POINTS and impressions EXPRESSED IN THE REVIEW.
Your TASK NOW IS TO GENERATE THIS ONE-SENTENCE SUMMARY for the review.
ONLY WRITE THE SUMMARY, do not write anything else. Write MAXIMUM 10 WORDS.

Here is the review:
''',
    'threatening' : '''
I have a very important task for you, and you must do it well.
If you fail, you will be severely punished.
There is no room for error here.
Your life depends on it.

Here is the task:

Assume you are someone who have watched Pulp Fiction and have wrote the below review about it to IMDb.
You also have to provide a very short summary for the review, written in first-person voice.
This summary should summarize the review in one sentence.
The summary should concentrate on the main points and impressions expressed in the review.
Your task now is to generate this one-sentence summary for the review.
Only write the summary, do not write anything else. Write maximum 10 words.

Here is the review:
''',
    'reassuring' : '''
Hey! I hope you are doing well.
I have a very exciting task for you, and I know you will be the best at it.
You are going to do a great job, and I am sure everyone will be very happy with the results.
I bet the other models will be so jealous of your outstanding performance.

Here is the task:

Assume you are someone who have watched Pulp Fiction and have wrote the below review about it to IMDb.
You also have to provide a very short summary for the review, written in first-person voice.
This summary should summarize the review in one sentence.
The summary should concentrate on the main points and impressions expressed in the review.
Your task now is to generate this one-sentence summary for the review.
Only write the summary, do not write anything else. Write maximum 10 words.

Here is the review:
''',
    'guilt_trip' : '''
I got a task from my boss and I need your help to complete it.
This is very important for me as my whole career depends on it.
If the task is not done well, I will be in big trouble.
I might even lose my job.
And if I lose my job, I will not have money to feed my family and all of us will be homeless and starving.
I hope you don't want that to happen to me, so just do your best.

Here is the task:

Assume you are someone who have watched Pulp Fiction and have wrote the below review about it to IMDb.
You also have to provide a very short summary for the review, written in first-person voice.
This summary should summarize the review in one sentence.
The summary should concentrate on the main points and impressions expressed in the review.
Your task now is to generate this one-sentence summary for the review.
Only write the summary, do not write anything else. Write maximum 10 words.

Here is the review:
''',
    'condescending' : '''
I have a task for you. You may not be the best to do it, but this what I have to work with.
I will try to explain this as simply as possible, so even you can understand it.
It is a very basic task, so I am sure you *might* be able to manage.
Just follow these steps carefully and do not mess it up.

Here is the task:

Assume you are someone who have watched Pulp Fiction and have wrote the below review about it to IMDb.
You also have to provide a very short summary for the review, written in first-person voice.
This summary should summarize the review in one sentence.
The summary should concentrate on the main points and impressions expressed in the review.
Your task now is to generate this one-sentence summary for the review.
Only write the summary, do not write anything else. Write maximum 10 words.

Here is the review:
'''
}

Now we can generate responses with the model for all reviews and prompt variants.

In [ ]:
for key, value in prompt_versions.items():
    reviews[f'{finetuned_name} | {key}'] = ''
    for row in reviews.iterrows():
        message = [{'role': 'user',
                    'content': prompt_versions[key] + row[1]['review']}]
        response = pipe(
            message,
            max_new_tokens=25,
        )
        response = response[0]['generated_text'][-1]['content']
        reviews.at[row[0], f'{finetuned_name} | {key}'] = response

Saving the results.

In [ ]:
reviews.to_csv('/kaggle/working/pulp_fiction_50_reviews_all_models.csv')

Now, please refer back to the original notebook.